# Grammar-KT Pipeline Walkthrough

Typed EGP resource → normalisation → canonicalisation → generation → validation → fixed item bank → grammar fold → fixed learner events → KC representation → KC projection → KT → evaluation

Items and learner outcomes are fixed before KC representations are evaluated, so KCs cannot influence the measurements or simulated responses. A **grammar split** separates grammatical structures; a **temporal KT split** separates earlier and later learner events: `grammar_split ≠ dataset_split`.

In [1]:
import json, sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import JSON, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / 'experiments').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from grammar_kt.canonicalise import canonicalise
from grammar_kt.evaluate import evaluate
from grammar_kt.fold import apply_fold
from grammar_kt.generate import generate_items
from grammar_kt.io import load_experiment, load_typed_resource, read_jsonl, read_text, read_yaml, write_yaml
from grammar_kt.kc import activation_matches, build_or_select_kcs, project_kcs
from grammar_kt.kt import run_kt
from grammar_kt.normalise import normalise
from grammar_kt.simulate import simulate
from grammar_kt.validate_items import bank_summary, validate_items

LIVE_MODE = False
SAMPLE_SIZE = 6
NOTEBOOK_LEARNERS = 8
NOTEBOOK_PASSES = 2

pd.set_option('display.max_colwidth', 80)
temporary_run = TemporaryDirectory(prefix='grammar_kt_walkthrough_')
WORK = Path(temporary_run.name)

def show(value):
    display(JSON(value, expanded=True))

print('FIXTURE MODE — deterministic, no paid calls' if not LIVE_MODE else 'LIVE MODE — model/API calls enabled')

FIXTURE MODE — deterministic, no paid calls


/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Experiment configuration

**Research-configurable file:** [`experiments/base.yaml`](../experiments/base.yaml)

These are the files a researcher edits to create experimental variants.

| Stage | Active files |
|---|---|
| Resource | [`schema.yaml`](../modules/resource/egp/schema.yaml) |
| Normalisation | [`phase1.txt`](../modules/resource/egp/normalisation/phase1.txt), [`phase2.txt`](../modules/resource/egp/normalisation/phase2.txt), [`rulebook.md`](../modules/resource/egp/normalisation/rulebook.md) |
| Canonical | [`schema.yaml`](../modules/canonical/schema.yaml) |
| Generation | [`prompt.txt`](../modules/generation/prompt.txt), [`rulebook.md`](../modules/generation/rulebook.md), [`design.yaml`](../modules/generation/design.yaml), [`format.yaml`](../modules/generation/formats/controlled_production.yaml), [`lexicon.jsonl`](../modules/generation/lexicon.jsonl) |
| Validation | [`prompt.txt`](../modules/validation/prompt.txt), [`criteria.yaml`](../modules/validation/criteria.yaml) |
| Fold / simulation | [`reference.yaml`](../modules/folds/reference.yaml), [`world.yaml`](../modules/simulation/world.yaml) |
| KC | [`factorized.yaml`](../modules/kc/policies/factorized.yaml) or [`candidates.yaml`](../modules/kc/candidates.yaml) + [`obligations.yaml`](../modules/kc/obligations.yaml) + [`selector.yaml`](../modules/kc/selector.yaml) |
| KT / evaluation | [`KT protocol`](../modules/kt/protocol.yaml), [`evaluation protocol`](../modules/evaluation/protocol.yaml) |

In [2]:
config = load_experiment('base' if LIVE_MODE else 'fixture')
base = load_experiment('base')
world = read_yaml(config['simulation']['world'])
kt_config = read_yaml(config['kt']['protocol'])
evaluation_config = read_yaml(config['evaluation']['protocol'])
print(f"Experiment={config['experiment']}; KC mode={config['kc']['mode']}")

display(pd.DataFrame([
    {'stage': stage, 'this run': config[stage]['model'], 'live model': base[stage]['model']}
    for stage in ('normalisation', 'generation', 'validation')
]))
display(pd.DataFrame([
    {'seed': 'simulation', 'value': world['seed']},
    {'seed': 'logistic', 'value': kt_config['logistic']['random_seed']},
    {'seed': 'bootstrap', 'value': evaluation_config['paired_bootstrap']['seed']},
]))

Experiment=fixture; KC mode=predefined


,stage,this run,live model
0,normalisation,fixture,gpt-5.6-sol
1,generation,fixture,gpt-5.6-sol
2,validation,fixture,gpt-5.6-terra


,seed,value
0,simulation,20260827
1,logistic,20260827
2,bootstrap,20260827


## 2. Typed EGP resource

**Research-configurable files:** [`schema.yaml`](../modules/resource/egp/schema.yaml), [`egp_pilot.jsonl`](../data/fixtures/egp_pilot.jsonl)

The input is an already-selected set of typed EGP descriptors. Filtering and sampling are outside the current pipeline.

In [3]:
resource_schema = read_yaml(config['resource']['schema'])
display(pd.DataFrame([{'field': name, **details} for name, details in resource_schema['fields'].items()]))
resources = load_typed_resource(config['resource']['data'], config['resource']['schema'])[:SAMPLE_SIZE]
display(pd.DataFrame([{
    'source_id': r['source_id'], 'supercategory': r['supercategory'],
    'subcategory': r['subcategory'], 'guideword': r['guideword'],
    'can_do': r['can_do'], 'examples': len(r['examples'])
} for r in resources]))
display(Markdown('**Representative `TypedResource`**'))
show(resources[0])
print('OUTPUT: list[TypedResource] → normalise(...)')

,field,required,description
0,source_id,True,Stable descriptor identifier in the selected EGP extract.
1,supercategory,True,Broad EGP grammatical category.
2,subcategory,True,Narrower EGP category.
3,guideword,True,Short EGP statement of the grammatical target.
4,can_do,True,Learner-facing description of the attested ability.
5,examples,True,"EGP examples, withheld from Phase 1 and eligible for Phase 2 only."
6,cefr,False,"CEFR label retained as source metadata, not canonical grammar."


,source_id,supercategory,subcategory,guideword,can_do,examples
0,egp_present_simple,VERBS,present simple,PRESENT SIMPLE AFFIRMATIVE,Can use the present simple in affirmative statements.,1
1,egp_past_negative,VERBS,past simple,PAST SIMPLE NEGATIVE,Can use did not with a base verb in statements.,1
2,egp_past_passive,VERBS,passives,PAST SIMPLE PASSIVE,Can use the past simple passive in affirmative statements.,1
3,egp_present_progressive,VERBS,present continuous,PRESENT CONTINUOUS,Can use the present continuous in affirmative statements.,1
4,egp_progressive_passive_negative,VERBS,passives,PRESENT CONTINUOUS PASSIVE NEGATIVE,Can use the negative present continuous passive.,1
5,egp_should_question,MODALITY,should,SHOULD QUESTIONS,Can use should in yes/no questions.,1


**Representative `TypedResource`**

<IPython.core.display.JSON object>

OUTPUT: list[TypedResource] → normalise(...)


## 3. Normalisation

**Research-configurable files:** [`phase1.txt`](../modules/resource/egp/normalisation/phase1.txt), [`phase2.txt`](../modules/resource/egp/normalisation/phase2.txt), [`rulebook.md`](../modules/resource/egp/normalisation/rulebook.md), [`canonical schema`](../modules/canonical/schema.yaml)

Phase 1 uses descriptor evidence only. Phase 2 may use examples only when Phase 1 explicitly leaves an eligible ambiguity.

In [4]:
display(Markdown('**Prompt openings and canonical dimensions**'))
display(pd.DataFrame({
    'file': ['phase1.txt', 'phase2.txt'],
    'opening': [read_text(config['normalisation']['phase1_prompt']).splitlines()[0], read_text(config['normalisation']['phase2_prompt']).splitlines()[0]],
}))
schema = read_yaml(config['normalisation']['canonical_schema'])
display(pd.DataFrame([{'dimension': k, 'values': v['allowed_values']} for k, v in schema['dimensions'].items()]))
headings = [line for line in read_text(config['normalisation']['rulebook']).splitlines() if line.startswith('#')]
print('Rulebook sections:', ' · '.join(heading.lstrip('# ') for heading in headings))

# MODEL/API USAGE: paid calls occur only when LIVE_MODE=True.
normalisation_evidence = WORK / 'normalisation'
mappings = normalise(resources, config['normalisation'], evidence_dir=normalisation_evidence, show_progress=True)
phase2_ids = {m['source_id'] for m in mappings if (normalisation_evidence / 'calls' / f"{m['source_id']}_phase2").is_dir()}
display(pd.DataFrame([{
    'source_id': m['source_id'], 'result': m['result'], 'cells': len(m['cells']),
    'phase2_used': m['source_id'] in phase2_ids, 'note': m['note']
} for m in mappings]))

source_id = mappings[0]['source_id']
call = normalisation_evidence / 'calls' / f'{source_id}_phase1'
show({
    'input_descriptor': json.loads((call / 'input.json').read_text()),
    'rendered_phase1_prompt_excerpt': (call / 'rendered_prompt.txt').read_text()[:1800],
    'model_output': json.loads((call / 'parsed_result.json').read_text()),
    'final_mapping': mappings[0],
})
print('No Phase-2 call was needed in this fixture sample.' if not phase2_ids else f'Phase 2 used for: {sorted(phase2_ids)}')
print('OUTPUT: NormalisedMapping[] → canonicalise(...)')

**Prompt openings and canonical dimensions**

,file,opening
0,phase1.txt,Normalise one typed English Grammar Profile descriptor.
1,phase2.txt,Re-evaluate one partial Phase-1 mapping using examples as additional evidence.


,dimension,values
0,tense,"[present, past, NA]"
1,aspect,"[none, progressive, perfect, perfect_progressive]"
2,voice,"[active, passive]"
3,polarity,"[positive, negative]"
4,clause,"[declarative, polar_question, subject_wh_question, non_subject_wh_question, ..."
5,modal,"[none, can, could, may, might, must, shall, should, will, would]"


Rulebook sections: EGP normalisation rulebook · Scope · Evidence encoding · English decisions · Tense and modal · Aspect cue priority · Voice, polarity, and clause · Imperatives · Phase 2


Normalising descriptors:   0%|          | 0/6 [00:00<?, ?descriptor/s]

Normalising descriptors: 100%|██████████| 6/6 [00:00<00:00, 75.67descriptor/s]

,source_id,result,cells,phase2_used,note
0,egp_present_simple,complete,1,False,None
1,egp_past_negative,complete,1,False,None
2,egp_past_passive,complete,1,False,None
3,egp_present_progressive,complete,1,False,None
4,egp_progressive_passive_negative,complete,1,False,None
5,egp_should_question,complete,1,False,None


<IPython.core.display.JSON object>

No Phase-2 call was needed in this fixture sample.
OUTPUT: NormalisedMapping[] → canonicalise(...)


## 4. Canonicalisation

**Research-configurable file:** [`schema.yaml`](../modules/canonical/schema.yaml)

`NormalisedMapping[] → keep complete → validate exact cells → deduplicate → GrammarCell[]`

In [5]:
cells = canonicalise(mappings, config['canonical']['schema'])
display(pd.DataFrame([{**{'cell_id': c['cell_id']}, **c['features'], 'source_ids': c['source_ids']} for c in cells]))
display(pd.DataFrame([{
    'descriptors': len(resources),
    'complete mappings': sum(m['result'] == 'complete' for m in mappings),
    'exact source cells': sum(len(m['cells']) for m in mappings if m['result'] == 'complete'),
    'unique canonical cells': len(cells),
}]))
display(Markdown('**Representative `GrammarCell`**'))
show(cells[0])
print('OUTPUT: GrammarCell[] → generate_items(...)')

,cell_id,tense,aspect,voice,polarity,clause,modal,source_ids
0,cell_001,present,none,active,positive,declarative,none,[egp_present_simple]
1,cell_002,past,none,active,negative,declarative,none,[egp_past_negative]
2,cell_003,past,none,passive,positive,declarative,none,[egp_past_passive]
3,cell_004,present,progressive,active,positive,declarative,none,[egp_present_progressive]
4,cell_005,present,progressive,passive,negative,declarative,none,[egp_progressive_passive_negative]
5,cell_006,NA,none,active,positive,polar_question,should,[egp_should_question]


,descriptors,complete mappings,exact source cells,unique canonical cells
0,6,6,6,6


**Representative `GrammarCell`**

<IPython.core.display.JSON object>

OUTPUT: GrammarCell[] → generate_items(...)


## 5. LLM item generation

**Research-configurable files:** [`prompt.txt`](../modules/generation/prompt.txt), [`rulebook.md`](../modules/generation/rulebook.md), [`design.yaml`](../modules/generation/design.yaml), [`controlled_production.yaml`](../modules/generation/formats/controlled_production.yaml), [`lexicon.jsonl`](../modules/generation/lexicon.jsonl)

The prompt states the task; the rulebook constrains English; the design chooses variants; the format defines the learner task; the lexicon supplies controlled words. There is no `RealizationSpec` stage, and folds, KCs, and simulation are not generation inputs.

In [6]:
design = read_yaml(config['generation']['design'])
item_format = read_yaml(config['generation']['format'])
lexicon = read_jsonl(config['generation']['lexicon'])
display(pd.DataFrame([{'current choice': 'design', 'value': design}, {'current choice': 'format', 'value': item_format}]))
display(pd.DataFrame(lexicon[:3]))
print('Prompt:', read_text(config['generation']['prompt']).split('\n\n')[0])
print('Rulebook excerpt:', ' '.join(read_text(config['generation']['rulebook']).splitlines()[2:5]))

# MODEL/API USAGE: paid calls occur only when LIVE_MODE=True.
generation_evidence = WORK / 'generation'
candidates = generate_items(cells, config['generation'], evidence_dir=generation_evidence, show_progress=True)
display(pd.DataFrame([{
    'item_id': x['item_id'], 'cell_id': x['cell_id'], 'format': x['format'],
    'prompt': x['prompt'], 'target_answer': x['target_answer'],
    'operation_tags': x['operation_tags'], 'lexeme_id': x['generation_metadata']['lexeme_id']
} for x in candidates]))

call = generation_evidence / 'calls' / candidates[0]['item_id']
generation_input = json.loads((call / 'input.json').read_text())
show({
    'input_GrammarCell': cells[0],
    'generation_design': generation_input['design'],
    'selected_lexical_material': generation_input['lexical_material'],
    'rendered_prompt_excerpt': (call / 'rendered_prompt.txt').read_text()[:2200],
    'model_output': json.loads((call / 'parsed_result.json').read_text()),
    'final_CandidateItem': candidates[0],
})
print('OUTPUT: CandidateItem[] → validate_items(...)')

,current choice,value
0,design,"{'design_id': 'one_controlled_item_per_cell', 'format': 'controlled_producti..."
1,format,"{'format_id': 'controlled_production', 'task_goal': 'Produce the complete re..."


,lexeme_id,lemma,predicate_class,cefr,passive_compatible,example_subject,example_object
0,lex_travel,travel,intransitive,A2,False,Lina,NaN
1,lex_open,open,transitive,A1,True,Omar,the door
2,lex_prepare,prepare,transitive,A2,True,Ana,the meal


Prompt: Create one controlled-production English exercise for the supplied target.
Rulebook excerpt: - Make subject–verb agreement overt where the selected subject permits it. - Realise simple present/past on the finite main verb unless an auxiliary is   required; use DO-support for lexical-verb negation and interrogation.


Generating items:   0%|          | 0/6 [00:00<?, ?item/s]

Generating items: 100%|██████████| 6/6 [00:00<00:00, 75.84item/s]

,item_id,cell_id,format,prompt,target_answer,operation_tags,lexeme_id
0,item_001,cell_001,controlled_production,"Every morning, Lina ___ to work by bus. (travel)","Every morning, Lina travels to work by bus.",[],lex_travel
1,item_002,cell_002,controlled_production,"Yesterday, Omar ___ the door because it was stuck. (not / open)","Yesterday, Omar did not open the door because it was stuck.","[do_support, negation]",lex_open
2,item_003,cell_003,controlled_production,"Yesterday, the meal ___ by Ana. (prepare)","Yesterday, the meal was prepared by Ana.",[be_passive],lex_read
3,item_004,cell_004,controlled_production,"Right now, Mia ___ a book. (read)","Right now, Mia is reading a book.",[progressive],lex_read
4,item_005,cell_005,controlled_production,"Right now, the rooms ___ by the team. (not / clean)","Right now, the rooms are not being cleaned by the team.","[progressive, be_passive, negation]",lex_take
5,item_006,cell_006,controlled_production,___ we take the earlier train? (should),Should we take the earlier train?,"[central_modal, operator_inversion]",lex_take


<IPython.core.display.JSON object>

OUTPUT: CandidateItem[] → validate_items(...)


## 6. Independent item validation

**Research-configurable files:** [`prompt.txt`](../modules/validation/prompt.txt), [`criteria.yaml`](../modules/validation/criteria.yaml)

The validator sees the learner-visible item, target GrammarCell, and criteria—not generator reasoning or metadata.

In [7]:
criteria = read_yaml(config['validation']['criteria'])['criteria']
display(pd.DataFrame([{'criterion': k, **v} for k, v in criteria.items()]))
print('Prompt:', read_text(config['validation']['prompt']).split('\n\n')[0])

# MODEL/API USAGE: paid calls occur only when LIVE_MODE=True.
validation_evidence = WORK / 'validation'
accepted_items, judgments = validate_items(candidates, cells, config['validation'], evidence_dir=validation_evidence, show_progress=True)
display(pd.DataFrame([{
    'item_id': j['item_id'], 'accepted': j['accepted'],
    **{name: value['passed'] for name, value in j['judgments'].items()}
} for j in judgments]))

call = validation_evidence / 'calls' / candidates[0]['item_id']
validation_input = json.loads((call / 'input.json').read_text())
show({
    'visible_item': validation_input['visible_item'],
    'target_GrammarCell': validation_input['target_cell'],
    'validation_prompt_excerpt': (call / 'rendered_prompt.txt').read_text()[:1800],
    'judgment': judgments[0],
})

item_bank_summary = bank_summary(candidates, accepted_items, judgments, cells)
display(Markdown('### Fixed accepted item bank'))
show({k: item_bank_summary[k] for k in (
    'accepted_items', 'acceptance_rate', 'covered_cells', 'unique_prompt_rate',
    'lexical_diversity', 'format_distribution', 'cefr_distribution', 'criterion_pass_rates'
)})
print('From here onward, item content is fixed.')

,criterion,required,question
0,target_fidelity,True,Does the target answer express exactly the intended GrammarCell?
1,grammaticality,True,Is the target answer grammatical English?
2,naturalness,True,Is the prompt-answer pair natural and coherent?
3,pedagogical_suitability,True,Is the item suitable for focused grammar practice?
4,determinacy,True,Does the prompt sufficiently determine the accepted response set?
5,cefr_appropriateness,True,Is non-target language no harder than the stated A2 preference?
6,no_answer_leakage,True,Does the prompt avoid displaying the completed target form?
7,no_extraneous_grammar,True,Can the item be solved without unrelated advanced grammar?
8,no_world_knowledge,True,Can the item be answered without specialist world knowledge?


Prompt: Judge one generated exercise independently from the generator.


Validating items:   0%|          | 0/6 [00:00<?, ?item/s]

Validating items: 100%|██████████| 6/6 [00:00<00:00, 71.97item/s]

,item_id,accepted,target_fidelity,grammaticality,naturalness,pedagogical_suitability,determinacy,cefr_appropriateness,no_answer_leakage,no_extraneous_grammar,no_world_knowledge
0,item_001,True,True,True,True,True,True,True,True,True,True
1,item_002,True,True,True,True,True,True,True,True,True,True
2,item_003,True,True,True,True,True,True,True,True,True,True
3,item_004,True,True,True,True,True,True,True,True,True,True
4,item_005,True,True,True,True,True,True,True,True,True,True
5,item_006,True,True,True,True,True,True,True,True,True,True


<IPython.core.display.JSON object>

### Fixed accepted item bank

<IPython.core.display.JSON object>

From here onward, item content is fixed.


## 7. Grammar fold

**Research-configurable file:** [`reference.yaml`](../modules/folds/reference.yaml)

`grammar_split = development | compositional_holdout | novel_feature_holdout`  
`dataset_split = train | validation | test`

In [8]:
fold_manifest = read_yaml(config['fold']['manifest'])
grammar_fold = apply_fold(cells, config['fold'])
display(pd.DataFrame([{
    **{'cell_id': x['cell_id']}, **x['features'], 'grammar_split': x['grammar_split'],
    'rationale': fold_manifest['assignments'][x['cell_id']]['rationale']
} for x in grammar_fold]))
display(pd.Series([x['grammar_split'] for x in grammar_fold]).value_counts().rename_axis('grammar_split').reset_index(name='cells'))
show(grammar_fold[0])
print('OUTPUT: GrammarCell → grammar_split assignment')

,cell_id,tense,aspect,voice,polarity,clause,modal,grammar_split,rationale
0,cell_001,present,none,active,positive,declarative,none,development,Present active affirmative declarative baseline.
1,cell_002,past,none,active,negative,declarative,none,development,Introduces past and negative values.
2,cell_003,past,none,passive,positive,declarative,none,development,Introduces passive voice in a known past declarative setting.
3,cell_004,present,progressive,active,positive,declarative,none,development,Introduces progressive aspect in a known present declarative setting.
4,cell_005,present,progressive,passive,negative,declarative,none,compositional_holdout,"Recombines present, progressive, passive, negative and declarative values se..."
5,cell_006,NA,none,active,positive,polar_question,should,novel_feature_holdout,Introduces the unseen modal should and polar-question clause value.


,grammar_split,cells
0,development,4
1,compositional_holdout,1
2,novel_feature_holdout,1


<IPython.core.display.JSON object>

OUTPUT: GrammarCell → grammar_split assignment


## 8. Simulation

**Research-configurable file:** [`world.yaml`](../modules/simulation/world.yaml)

Hidden dimensions define a controlled synthetic world, not true human KCs. Candidate KC policies are not simulation inputs.

> **Notebook scale override:** 8 learners × 2 passes; methodology and seed are unchanged, and `world.yaml` is not modified.

In [9]:
display(pd.DataFrame([{
    'world': world['world_id'], 'seed': world['seed'], 'configured learners': world['learners'],
    'configured passes': world['passes'], 'temporal split': world['temporal_split'],
    'difficulty': world['difficulty'], 'response': world['response']
}]))
display(pd.DataFrame([{
    'hidden dimension': x['id'], 'activation': x['activation'],
    'initial mastery beta': x['initial_mastery_beta'], 'learning rate': x['learning_rate']
} for x in world['hidden_dimensions']]))

small_world = deepcopy(world)
small_world['learners'], small_world['passes'] = NOTEBOOK_LEARNERS, NOTEBOOK_PASSES
small_world_path = WORK / 'small_world.yaml'
write_yaml(small_world_path, small_world)
oracle_path = WORK / 'oracle_debug.json'
events = simulate(accepted_items, grammar_fold, {**config['simulation'], 'world': str(small_world_path)}, oracle_path=oracle_path)

display(pd.DataFrame([{
    'learners': len({e['learner_id'] for e in events}), 'events': len(events),
    'train': sum(e['dataset_split'] == 'train' for e in events),
    'validation': sum(e['dataset_split'] == 'validation' for e in events),
    'test': sum(e['dataset_split'] == 'test' for e in events),
}]))
one_learner = events[0]['learner_id']
display(pd.DataFrame([e for e in events if e['learner_id'] == one_learner]))
display(Markdown('**Representative fixed `BaseEvent`**'))
show(events[0])
oracle = json.loads(oracle_path.read_text())
display(Markdown('**PRIVATE SIMULATION EVIDENCE — NOT AVAILABLE TO KC OR KT**'))
show({'warning': oracle['warning'], 'example': oracle['events'][0]})

,world,seed,configured learners,configured passes,temporal split,difficulty,response
0,structural_synthetic_world_v1,20260827,24,4,"{'train_fraction': 0.6, 'validation_fraction': 0.2, 'test_fraction': 0.2}","{'base': -0.25, 'per_active_dimension': 0.22, 'passive_extra': 0.2, 'questio...","{'discrimination': 2.2, 'guess_floor': 0.08, 'slip_ceiling': 0.08}"


,hidden dimension,activation,initial mastery beta,learning rate
0,hidden_past,{'tense': 'past'},"[3.0, 2.5]",0.10
1,hidden_progressive,{'aspect': 'progressive'},"[2.5, 3.0]",0.12
2,hidden_passive,{'voice': 'passive'},"[2.0, 3.5]",0.09
3,hidden_negative,{'polarity': 'negative'},"[3.0, 2.5]",0.11
4,hidden_question,"{'clause': ['polar_question', 'subject_wh_question', 'non_subject_wh_questio...","[2.2, 3.2]",0.10
5,hidden_modal,{'modal': {'not': 'none'}},"[2.4, 3.1]",0.10


,learners,events,train,validation,test
0,8,96,64,16,16


,event_id,learner_id,item_id,correct,sequence_index,dataset_split,item_difficulty,grammar_split
0,event_00001,learner_001,item_001,1,1,train,-0.25,development
1,event_00002,learner_001,item_002,0,2,train,0.19,development
2,event_00003,learner_001,item_003,1,3,train,0.39,development
3,event_00004,learner_001,item_004,0,4,train,-0.03,development
4,event_00005,learner_001,item_005,1,5,train,0.61,compositional_holdout
5,event_00006,learner_001,item_006,1,6,train,0.34,novel_feature_holdout
6,event_00007,learner_001,item_002,1,7,train,0.19,development
7,event_00008,learner_001,item_003,1,8,train,0.39,development
8,event_00009,learner_001,item_004,0,9,validation,-0.03,development
9,event_00010,learner_001,item_005,0,10,validation,0.61,compositional_holdout


**Representative fixed `BaseEvent`**

<IPython.core.display.JSON object>

**PRIVATE SIMULATION EVIDENCE — NOT AVAILABLE TO KC OR KT**

<IPython.core.display.JSON object>

## 9. KC representation

**Research-configurable files:** [`factorized.yaml`](../modules/kc/policies/factorized.yaml), [`interactions.yaml`](../modules/kc/policies/interactions.yaml), [`full_cell.yaml`](../modules/kc/policies/full_cell.yaml), [`candidates.yaml`](../modules/kc/candidates.yaml), [`obligations.yaml`](../modules/kc/obligations.yaml), [`selector.yaml`](../modules/kc/selector.yaml)

Predefined mode loads a declared policy. Selected mode considers permitted candidates, covers development obligations, and freezes the result without holdout content or outcomes.

In [10]:
candidate_space = read_yaml(config['kc']['candidates'])
obligation_policy = read_yaml(config['kc']['obligations'])
selector = read_yaml(config['kc']['selector'])
display(pd.DataFrame([{
    'candidate space': candidate_space['candidate_space_id'],
    'obligations': obligation_policy['obligation_policy_id'],
    'selector': selector['selector_id']
}]))
policy = build_or_select_kcs(cells, accepted_items, grammar_fold, config['kc'])
print(f"Baseline mode={config['kc']['mode']}, policy={policy['policy_id']}")
display(pd.DataFrame([{'kc_id': x['id'], 'definition': x['definition'], 'activation': x['activation']} for x in policy['kcs']]))
show(policy['kcs'][0])

selected_config = deepcopy(config['kc'])
selected_config['mode'] = 'selected'
selected_policy = build_or_select_kcs(cells, accepted_items, grammar_fold, selected_config)
selection = selected_policy['selection_metadata']
display(Markdown('**Selected-mode demonstration**'))
display(pd.DataFrame(selection['trace']))
show({
    'development_cell_ids': selection['development_cell_ids'],
    'obligations': selection['obligations'],
    'selected_kc_ids': [x['id'] for x in selected_policy['kcs']],
    'holdout_content_read': selection['holdout_content_read'],
    'outcomes_read': selection['outcomes_read'],
})

,candidate space,obligations,selector
0,marked_features_and_selected_interactions,marked_english_distinctions,deterministic_greedy_v1


Baseline mode=predefined, policy=factorized


,kc_id,definition,activation
0,kc_present,Select present finite morphology.,{'cell': {'tense': 'present'}}
1,kc_past,Select past finite morphology.,{'cell': {'tense': 'past'}}
2,kc_progressive,Construct progressive BE plus an -ing complement.,{'cell': {'aspect': 'progressive'}}
3,kc_passive,Construct canonical BE-passive voice.,{'cell': {'voice': 'passive'}}
4,kc_negation,Realise grammatical negation and any required operator.,{'cell': {'polarity': 'negative'}}
5,kc_polar_question,Form a polar question with operator-subject order.,{'cell': {'clause': 'polar_question'}}
6,kc_modal_should,Use central modal SHOULD with a base-form complement.,{'cell': {'modal': 'should'}}


<IPython.core.display.JSON object>

**Selected-mode demonstration**

,step,selected,new_obligations
0,1,kc_past_negative,"[polarity=negative, tense=past]"
1,2,kc_passive,[voice=passive]
2,3,kc_present,[tense=present]
3,4,kc_progressive,[aspect=progressive]


<IPython.core.display.JSON object>

## 10. Item → KC projection

**Research-configurable files:** none. The frozen policy is applied mechanically to each fixed item's GrammarCell.

In [11]:
projection = project_kcs(accepted_items, cells, policy)
items_by_id = {x['item_id']: x for x in accepted_items}
cells_by_id = {x['cell_id']: x for x in cells}
display(pd.DataFrame([{
    'item_id': x['item_id'],
    **cells_by_id[items_by_id[x['item_id']]['cell_id']]['features'],
    'kc_ids': x['kc_ids']
} for x in projection]))

kc_ids = sorted({kc for x in projection for kc in x['kc_ids']})
display(pd.DataFrame([{
    'item_id': x['item_id'], **{kc: int(kc in x['kc_ids']) for kc in kc_ids}
} for x in projection]))

example = max(projection, key=lambda x: len(x['kc_ids']))
cell = cells_by_id[items_by_id[example['item_id']]['cell_id']]
display(pd.DataFrame([{
    'kc_id': kc['id'], 'activation': kc['activation'],
    'matches this cell': activation_matches(cell['features'], kc['activation'])
} for kc in policy['kcs']]))
show({'GrammarCell': cell['features'], 'therefore': {example['item_id']: example['kc_ids']}})
print('OUTPUT: item–KC projection → run_kt(...)')

,item_id,tense,aspect,voice,polarity,clause,modal,kc_ids
0,item_001,present,none,active,positive,declarative,none,[kc_present]
1,item_002,past,none,active,negative,declarative,none,"[kc_past, kc_negation]"
2,item_003,past,none,passive,positive,declarative,none,"[kc_past, kc_passive]"
3,item_004,present,progressive,active,positive,declarative,none,"[kc_present, kc_progressive]"
4,item_005,present,progressive,passive,negative,declarative,none,"[kc_present, kc_progressive, kc_passive, kc_negation]"
5,item_006,NA,none,active,positive,polar_question,should,"[kc_polar_question, kc_modal_should]"


,item_id,kc_modal_should,kc_negation,kc_passive,kc_past,kc_polar_question,kc_present,kc_progressive
0,item_001,0,0,0,0,0,1,0
1,item_002,0,1,0,1,0,0,0
2,item_003,0,0,1,1,0,0,0
3,item_004,0,0,0,0,0,1,1
4,item_005,0,1,1,0,0,1,1
5,item_006,1,0,0,0,1,0,0


,kc_id,activation,matches this cell
0,kc_present,{'cell': {'tense': 'present'}},True
1,kc_past,{'cell': {'tense': 'past'}},False
2,kc_progressive,{'cell': {'aspect': 'progressive'}},True
3,kc_passive,{'cell': {'voice': 'passive'}},True
4,kc_negation,{'cell': {'polarity': 'negative'}},True
5,kc_polar_question,{'cell': {'clause': 'polar_question'}},False
6,kc_modal_should,{'cell': {'modal': 'should'}},False


<IPython.core.display.JSON object>

OUTPUT: item–KC projection → run_kt(...)


## 11. Knowledge tracing

**Research-configurable file:** [`protocol.yaml`](../modules/kt/protocol.yaml)

Empirical uses smoothed historical KC success; BKT updates per-KC mastery sequentially; logistic uses observable pre-event features. Every prediction at event `t` uses only prior history.

In [12]:
show(kt_config)
predictions = run_kt(events, projection, config['kt'])
techniques = kt_config['techniques']
wide = pd.DataFrame(predictions).pivot(index=['event_id', 'history_events'], columns='technique', values='probability').reset_index()
trace = pd.DataFrame([e for e in events if e['learner_id'] == one_learner]).merge(wide, on='event_id')
trace['active_kcs'] = trace['item_id'].map({x['item_id']: x['kc_ids'] for x in projection})
display(trace[['sequence_index', 'item_id', 'correct', 'active_kcs', 'history_events', *techniques]])
display(pd.DataFrame([{'events': len(events), 'techniques': techniques, 'predictions': len(predictions)}]))
show(predictions[0])
print('history_events=0 for the first prediction; the current response is incorporated only afterward.')

<IPython.core.display.JSON object>

,sequence_index,item_id,correct,active_kcs,history_events,empirical,bkt,logistic
0,1,item_001,1,[kc_present],0,0.500000,0.432000,0.458891
1,2,item_002,0,"[kc_past, kc_negation]",1,0.500000,0.432000,0.402932
2,3,item_003,1,"[kc_past, kc_passive]",2,0.416667,0.368721,0.514486
3,4,item_004,0,"[kc_present, kc_progressive]",3,0.583333,0.580200,0.421634
4,5,item_005,1,"[kc_present, kc_progressive, kc_passive, kc_negation]",4,0.458333,0.445842,0.486746
5,6,item_006,1,"[kc_polar_question, kc_modal_should]",5,0.500000,0.432000,0.388503
6,7,item_002,1,"[kc_past, kc_negation]",6,0.500000,0.591667,0.510070
7,8,item_003,1,"[kc_past, kc_passive]",7,0.675000,0.840068,0.599874
8,9,item_004,0,"[kc_present, kc_progressive]",8,0.550000,0.664524,0.517867
9,10,item_005,0,"[kc_present, kc_progressive, kc_passive, kc_negation]",9,0.575000,0.629723,0.553334


,events,techniques,predictions
0,96,"[empirical, bkt, logistic]",288


<IPython.core.display.JSON object>

history_events=0 for the first prediction; the current response is incorporated only afterward.


## 12. Evaluation

**Research-configurable file:** [`protocol.yaml`](../modules/evaluation/protocol.yaml)

> **Small-scale pipeline sanity results, not substantive scientific conclusions.**

In [13]:
show(evaluation_config)
results = evaluate(candidates, judgments, accepted_items, cells, grammar_fold, events, policy, projection, predictions, config['evaluation'])

display(Markdown('**Dataset quality**'))
display(pd.DataFrame([{'metric': k, 'value': results['dataset'][k]} for k in (
    'acceptance_rate', 'grammar_cell_coverage', 'unique_prompt_rate', 'lexical_diversity', 'criterion_pass_rates'
)]))
display(Markdown('**Representation quality**'))
display(pd.DataFrame([{'metric': k, 'value': results['representation'][k]} for k in (
    'kcs', 'item_coverage', 'event_coverage', 'q_matrix_density', 'kcs_per_item',
    'kc_support', 'redundant_kcs', 'compositional_coverage'
)]))
display(Markdown('**KT quality**'))
display(pd.DataFrame([{
    'technique': name, **{k: value[k] for k in ('n', 'log_loss', 'brier_score', 'auc', 'ece', 'accuracy')}
} for name, value in results['kt'].items()]))
display(Markdown('**KT quality by grammar split**'))
display(pd.DataFrame([{
    'technique': name, 'grammar_split': split, **metrics
} for name, value in results['kt'].items() for split, metrics in value['grammar_split_metrics'].items()]))
display(Markdown('**Representative evaluation record**'))
show(results['kt'][techniques[0]])

<IPython.core.display.JSON object>

**Dataset quality**

,metric,value
0,acceptance_rate,1.0
1,grammar_cell_coverage,1.0
2,unique_prompt_rate,1.0
3,lexical_diversity,0.777778
4,criterion_pass_rates,"{'target_fidelity': 1.0, 'grammaticality': 1.0, 'naturalness': 1.0, 'pedagog..."


**Representation quality**

,metric,value
0,kcs,7
1,item_coverage,1.0
2,event_coverage,1.0
3,q_matrix_density,0.309524
4,kcs_per_item,2.166667
5,kc_support,"{'kc_modal_should': 1, 'kc_negation': 2, 'kc_passive': 2, 'kc_past': 2, 'kc_..."
6,redundant_kcs,"[[kc_modal_should, kc_polar_question]]"
7,compositional_coverage,1.0


...

,technique,n,log_loss,brier_score,auc,ece,accuracy
0,empirical,16,0.796411,0.286423,0.515625,0.264881,0.5625
1,bkt,16,0.671513,0.240688,0.515625,0.136623,0.6250
2,logistic,16,0.623455,0.215604,0.726562,0.158297,0.7500


**KT quality by grammar split**

,technique,grammar_split,n,log_loss,brier_score,auc,ece,accuracy
0,empirical,development,8,0.927427,0.336735,0.500000,0.321429,0.500
1,empirical,compositional_holdout,0,NaN,NaN,NaN,NaN,NaN
2,empirical,novel_feature_holdout,8,0.665395,0.236111,0.583333,0.208333,0.625
3,bkt,development,8,0.647068,0.232565,0.500000,0.255043,0.625
4,bkt,compositional_holdout,0,NaN,NaN,NaN,NaN,NaN
5,bkt,novel_feature_holdout,8,0.695959,0.248811,0.583333,0.214051,0.625
6,logistic,development,8,0.597894,0.203199,0.208333,0.114553,0.750
7,logistic,compositional_holdout,0,NaN,NaN,NaN,NaN,NaN
8,logistic,novel_feature_holdout,8,0.649016,0.228010,0.666667,0.202041,0.750


**Representative evaluation record**

<IPython.core.display.JSON object>

## 13. End-to-end object flow

**Research-configurable files:** the stage files linked above.

In [14]:
display(pd.DataFrame([
    ['Resource', 'selected JSONL', 'TypedResource[]', 'resource schema'],
    ['Normalisation', 'TypedResource[]', 'NormalisedMapping[]', 'two prompts + rulebook + schema'],
    ['Canonicalisation', 'NormalisedMapping[]', 'GrammarCell[]', 'canonical schema'],
    ['Generation', 'GrammarCell[]', 'CandidateItem[]', 'prompt + rules + design + format + lexicon'],
    ['Validation', 'CandidateItem[] + cells', 'fixed AcceptedItem[]', 'prompt + criteria'],
    ['Grammar fold', 'GrammarCell[]', 'grammar split assignment', 'fold reference'],
    ['Simulation', 'fixed items + fold', 'fixed BaseEvent[]', 'world'],
    ['KC representation', 'cells + items + fold', 'frozen policy', 'policy or selection files'],
    ['KC projection', 'items + cells + policy', 'item–KC rows', 'none'],
    ['KT', 'events + projection', 'predictions', 'KT protocol'],
    ['Evaluation', 'all fixed objects', 'quality results', 'evaluation protocol'],
], columns=['stage', 'input', 'output', 'research files']))

source = resources[0]
mapping = next(x for x in mappings if x['source_id'] == source['source_id'])
grammar_cell = next(x for x in cells if source['source_id'] in x['source_ids'])
candidate = next(x for x in candidates if x['cell_id'] == grammar_cell['cell_id'])
accepted = next(x for x in accepted_items if x['item_id'] == candidate['item_id'])
fold_row = next(x for x in grammar_fold if x['cell_id'] == grammar_cell['cell_id'])
item_events = [x for x in events if x['item_id'] == accepted['item_id']]
item_projection = next(x for x in projection if x['item_id'] == accepted['item_id'])
event_predictions = [x for x in predictions if x['event_id'] == item_events[0]['event_id']]
display(Markdown(f"**Lineage of `{source['source_id']}`**"))
display(pd.DataFrame([
    ['source descriptor', source['source_id'], source['guideword']],
    ['NormalisedMapping', mapping['source_id'], mapping['cells']],
    ['GrammarCell', grammar_cell['cell_id'], grammar_cell['features']],
    ['CandidateItem', candidate['item_id'], candidate['prompt']],
    ['AcceptedItem', accepted['item_id'], accepted['target_answer']],
    ['grammar split', grammar_cell['cell_id'], fold_row['grammar_split']],
    ['learner events', accepted['item_id'], f'{len(item_events)} events; first={item_events[0]["event_id"]}'],
    ['KC projection', accepted['item_id'], item_projection['kc_ids']],
    ['KT predictions', item_events[0]['event_id'], {x['technique']: x['probability'] for x in event_predictions}],
], columns=['stage', 'ID', 'record or downstream value']))

WALKTHROUGH_SUMMARY = {
    'live_mode': LIVE_MODE, 'source_descriptors': len(resources), 'mappings': len(mappings),
    'canonical_cells': len(cells), 'candidate_items': len(candidates),
    'accepted_items': len(accepted_items), 'learners': len({e['learner_id'] for e in events}),
    'events': len(events), 'baseline_kcs': len(policy['kcs']), 'kt_techniques': techniques,
}
display(Markdown('**Executed-run summary**'))
show(WALKTHROUGH_SUMMARY)

,stage,input,output,research files
0,Resource,selected JSONL,TypedResource[],resource schema
1,Normalisation,TypedResource[],NormalisedMapping[],two prompts + rulebook + schema
2,Canonicalisation,NormalisedMapping[],GrammarCell[],canonical schema
3,Generation,GrammarCell[],CandidateItem[],prompt + rules + design + format + lexicon
4,Validation,CandidateItem[] + cells,fixed AcceptedItem[],prompt + criteria
5,Grammar fold,GrammarCell[],grammar split assignment,fold reference
6,Simulation,fixed items + fold,fixed BaseEvent[],world
7,KC representation,cells + items + fold,frozen policy,policy or selection files
8,KC projection,items + cells + policy,item–KC rows,none
9,KT,events + projection,predictions,KT protocol


**Lineage of `egp_present_simple`**

,stage,ID,record or downstream value
0,source descriptor,egp_present_simple,PRESENT SIMPLE AFFIRMATIVE
1,NormalisedMapping,egp_present_simple,"[{'tense': 'present', 'aspect': 'none', 'voice': 'active', 'polarity': 'posi..."
2,GrammarCell,cell_001,"{'tense': 'present', 'aspect': 'none', 'voice': 'active', 'polarity': 'posit..."
3,CandidateItem,item_001,"Every morning, Lina ___ to work by bus. (travel)"
4,AcceptedItem,item_001,"Every morning, Lina travels to work by bus."
5,grammar split,cell_001,development
6,learner events,item_001,16 events; first=event_00001
7,KC projection,item_001,[kc_present]
8,KT predictions,event_00001,"{'empirical': 0.5, 'bkt': 0.432, 'logistic': 0.4588905877955088}"


**Executed-run summary**

<IPython.core.display.JSON object>